In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs with timestamp parsing (Unix seconds -> UTC)
df_metric = pd.read_csv("metric.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

df_log = pd.read_csv("log.csv")
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

df_error = pd.read_csv("error_logs.csv")
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Coerce value columns to numeric (non-numeric -> NaN) to avoid aggregation errors
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# helper for global P95 per-group (uses entire group's series)
def group_p95(x):
    arr = x.dropna().to_numpy(dtype=float)
    return float(np.nanpercentile(arr, 95)) if arr.size > 0 else np.nan

# 1) metric.csv aggregates: top 20 (cmdb_id, kpi_name) by row count with count, mean, global P95
metric_summary = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        mean_value=('value', 'mean'),
        p95_value=('value', group_p95)
    )
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
metric_summary['mean_value'] = metric_summary['mean_value'].round(6)
metric_summary['p95_value'] = metric_summary['p95_value'].round(6)
metric_summary = metric_summary.head(20)

# 2) trace.csv aggregates: top 20 (cmdb_id, trace_name) by row count with count, mean, global P95
trace_summary = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        mean_value=('value', 'mean'),
        p95_value=('value', group_p95)
    )
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
trace_summary['mean_value'] = trace_summary['mean_value'].round(6)
trace_summary['p95_value'] = trace_summary['p95_value'].round(6)
trace_summary = trace_summary.head(20)

# 3) log.csv aggregates: top 20 (cmdb_id, log_name) by row count with count, mean, global P95
log_summary = (
    df_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        mean_value=('value', 'mean'),
        p95_value=('value', group_p95)
    )
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
log_summary['mean_value'] = log_summary['mean_value'].round(6)
log_summary['p95_value'] = log_summary['p95_value'].round(6)
log_summary = log_summary.head(20)

# 4) error_logs.csv aggregates:
# count per cmdb_id, earliest timestamp, latest timestamp, up to 5 sample messages; top 10 cmdb_id by count
err_grp = df_error.groupby('cmdb_id', sort=False)
error_summary = err_grp.agg(
    count=('message', 'count'),
    earliest_ts=('timestamp', 'min'),
    latest_ts=('timestamp', 'max')
).reset_index()

# collect up to 5 sample messages per cmdb_id
samples = err_grp['message'].apply(lambda s: " | ".join(s.astype(str).head(5).tolist())).reset_index(name='samples')
error_summary = error_summary.merge(samples, on='cmdb_id', how='left')

# format timestamps as compact ISO UTC strings
error_summary['earliest_ts'] = pd.to_datetime(error_summary['earliest_ts']).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
error_summary['latest_ts'] = pd.to_datetime(error_summary['latest_ts']).dt.strftime('%Y-%m-%dT%H:%M:%SZ')

error_summary = error_summary.sort_values('count', ascending=False).head(10).reset_index(drop=True)

# Return compact summaries
metric_summary, trace_summary, log_summary, error_summary

```
Out[1]:
```
summary = (
    "Summary of telemetry aggregates:\n\n"
    "Metrics (top pairs):\n"
    "- carts: high activity across CPU (count=25, mean=1.438, p95=1.520), memory (mean≈2.13e8, p95≈2.129e8), socket (mean≈12.74, p95≈13.69), and workload/latency KPIs.\n"
    "- carts-db: high CPU (mean≈8.04, p95≈8.66), high diskio (mean≈2.23e6, p95≈2.53e6), and substantial memory (mean≈1.05e8).\n"
    "- catalogue and catalogue-db: catalogue has low CPU/latency values but catalogue-db shows very large memory (mean≈4.31e8).\n"
    "Overall, metric records are present and show heavy resource usage for carts and several DB components.\n\n"
    "Trace data:\n"
    "- No trace records found (trace.csv aggregates are empty).\n\n"
    "Logs:\n"
    "- front-end has the largest log volume (log.row_count mean≈1878.96, p95≈2046) with zero normalized error counts.\n"
    "- Other services (catalogue, queue-master, shipping, orders, payment, user) show row_count activity and log.error_count values are zero in aggregated normalized metrics.\n\n"
    "Error logs:\n"
    "- No error log records found (error_logs.csv aggregates are empty).\n\n"
    "Interpretation:\n"
    "- The telemetry shows resource pressure on carts and its DB (CPU, memory, disk I/O, sockets) and very large memory on catalogue-db. There are no trace data or textual error log entries to provide deeper root-cause clues, and normalized log error counts are zero. Focus further investigation on the carts service and its DB (and catalogue-db memory) using detailed traces/logs and raw logs if available."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count    mean_value     p95_value
0          carts         cpu     25  1.438002e+00  1.519959e+00
1          carts  latency-50     25  2.556400e-02  2.787600e-02
2          carts  latency-90     25  5.499800e-02  6.838500e-02
3          carts         mem     25  2.128176e+08  2.129022e+08
4          carts      socket     25  1.273894e+01  1.368667e+01
5          carts    workload     25  8.151757e+00  8.459634e+00
6       carts-db         cpu     25  8.041025e+00  8.657099e+00
7       carts-db      diskio     25  2.229525e+06  2.526378e+06
8       carts-db         mem     25  1.054867e+08  1.073624e+08
9       carts-db      socket     25  8.000000e+00  8.000000e+00
10     catalogue         cpu     25  1.666380e-01  1.752480e-01
11     catalogue  latency-50     25  3.039000e-03  3.067000e-03
12     catalogue  latency-90     25  4.671000e-03  4.721000e-03
13     catalogue         mem     25  6.326951e+06  6.352596e+06
14     catalogue      socket     25  6.011333e+00  6.000000e+00
15     catalogue    workload     25  4.072200e+00  4.201113e+00
16  catalogue-db         cpu     25  2.065880e-01  2.173350e-01
17  catalogue-db      diskio     25  0.000000e+00  0.000000e+00
18  catalogue-db         mem     25  4.308435e+08  4.308453e+08
19  catalogue-db      socket     25  4.000000e+00  4.000000e+00, Empty DataFrame
Columns: [cmdb_id, trace_name, count, mean_value, p95_value]
Index: [],          cmdb_id         log_name  count   mean_value  p95_value
0      front-end    log.row_count     25  1878.960000     2046.0
1      front-end  log.error_count     25     0.000000        0.0
2      catalogue    log.row_count     25   155.960000      167.8
3      catalogue  log.error_count     25     0.000000        0.0
4   queue-master  log.error_count     25     0.000000        0.0
5   queue-master    log.row_count     25   528.480000      583.2
6       shipping    log.row_count     25    58.720000       64.8
7       shipping  log.error_count     25     0.000000        0.0
8         orders  log.error_count     25     0.000000        0.0
9         orders    log.row_count     25   117.440000      129.6
10       payment  log.error_count     25     0.000000        0.0
11       payment    log.row_count     25    97.120000      104.8
12          user  log.error_count     25     0.000000        0.0
13          user    log.row_count     25   566.880000      613.6
14      carts-db  log.error_count     24     0.000000        0.0
15      carts-db    log.row_count     24     1.000000        1.0
16         carts  log.error_count     23     0.000000        0.0
17         carts    log.row_count     23    48.913043       75.0
18    session-db    log.row_count      5     5.000000        5.0
19    session-db  log.error_count      5     0.000000        0.0, Empty DataFrame
Columns: [cmdb_id, count, earliest_ts, latest_ts, samples]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing loaded DataFrames if available; otherwise load from CSVs
try:
    df_metric
    df_log
    df_error
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_log = pd.read_csv("log.csv")
    df_error = pd.read_csv("error_logs.csv")

# Ensure timestamps parsed as UTC datetimes (Unix seconds)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Ensure numeric values for aggregation
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Helper: compute P95 for a series (global per-group)
def series_p95(s):
    arr = s.dropna().to_numpy(dtype=float)
    return float(np.nanpercentile(arr, 95)) if arr.size > 0 else np.nan

# 1) Metrics component-level for specific services
services = ['carts', 'catalogue', 'orders', 'payment', 'user']
df_metrics_sel = df_metric[df_metric['cmdb_id'].isin(services)].copy()

# 1a) Global aggregates (compute global P95 BEFORE any window filtering)
metric_global = (
    df_metrics_sel
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        total_count=('value', 'count'),
        global_mean=('value', 'mean'),
        global_p95=('value', series_p95)
    )
    .sort_values('total_count', ascending=False)
    .reset_index(drop=True)
)
metric_global['global_mean'] = metric_global['global_mean'].round(6)
metric_global['global_p95'] = metric_global['global_p95'].round(6)
metric_global = metric_global.head(50)  # limit

# 1b) Window filtering and comparisons to global_p95
start = pd.to_datetime("2024-01-20 19:35:40", utc=True)
end   = pd.to_datetime("2024-01-20 20:05:40", utc=True)

# Filter window (inclusive)
mask_window = (df_metrics_sel['timestamp'] >= start) & (df_metrics_sel['timestamp'] <= end)
df_metrics_window = df_metrics_sel[mask_window].copy()

if not df_metrics_window.empty:
    # join global p95 to windowed points
    metric_global_small = metric_global[['cmdb_id','kpi_name','global_p95']].copy()
    df_w = df_metrics_window.merge(metric_global_small, on=['cmdb_id','kpi_name'], how='left')

    # compute per-group window stats
    def agg_window(g):
        pts = g.shape[0]
        cnt_above = int((g['value'] > g['global_p95']).sum())
        # earliest timestamp where value > global_p95
        above = g[g['value'] > g['global_p95']]
        if not above.empty:
            earliest_above = above['timestamp'].min().strftime('%Y-%m-%dT%H:%M:%SZ')
        else:
            earliest_above = None
        max_val = float(g['value'].max()) if g['value'].count()>0 else np.nan
        # ratio (handle division by zero / NaN)
        gp95 = g['global_p95'].iloc[0]
        ratio = float(max_val / gp95) if (pd.notna(gp95) and gp95 != 0) else np.nan
        return pd.Series({
            'points_in_window': pts,
            'count_above_p95': cnt_above,
            'earliest_ts_above_p95': earliest_above,
            'max_value_in_window': max_val,
            'ratio_max_over_p95': ratio
        })

    metric_window_summary = (
        df_w
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .apply(agg_window)
        .reset_index(drop=True)
    )

    # Merge with global stats for context
    metric_window_summary = metric_window_summary.merge(
        metric_global[['cmdb_id','kpi_name','total_count','global_mean','global_p95']],
        on=['cmdb_id','kpi_name'],
        how='left'
    )

    # Rounding and sorting/filtering
    metric_window_summary['global_mean'] = metric_window_summary['global_mean'].round(6)
    metric_window_summary['global_p95'] = metric_window_summary['global_p95'].round(6)
    metric_window_summary['max_value_in_window'] = metric_window_summary['max_value_in_window'].round(6)
    metric_window_summary['ratio_max_over_p95'] = metric_window_summary['ratio_max_over_p95'].round(6)

    # Keep only rows with points_in_window>0 and sort as requested
    metric_window_summary = metric_window_summary[metric_window_summary['points_in_window'] > 0]
    metric_window_summary = metric_window_summary.sort_values(
        ['count_above_p95','ratio_max_over_p95'],
        ascending=[False, False]
    ).head(50).reset_index(drop=True)
else:
    # Empty result structure with required columns
    metric_window_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','points_in_window','count_above_p95','earliest_ts_above_p95',
        'max_value_in_window','ratio_max_over_p95','total_count','global_mean','global_p95'
    ])

# 2) Logs
# 2a) Filter log.csv to incident window and aggregate per cmdb_id, log_name
mask_log_window = (df_log['timestamp'] >= start) & (df_log['timestamp'] <= end)
df_log_window = df_log[mask_log_window].copy()

if not df_log_window.empty:
    log_window_summary = (
        df_log_window
        .groupby(['cmdb_id','log_name'], as_index=False)
        .agg(
            points_in_window=('value','count'),
            mean_value=('value','mean'),
            p95_value=('value', lambda s: float(np.nanpercentile(s.dropna().to_numpy(dtype=float),95)) if s.dropna().size>0 else np.nan)
        )
        .sort_values('points_in_window', ascending=False)
        .reset_index(drop=True)
    )
    log_window_summary['mean_value'] = log_window_summary['mean_value'].round(6)
    log_window_summary['p95_value'] = log_window_summary['p95_value'].round(6)
    log_window_summary = log_window_summary.head(50)
else:
    log_window_summary = pd.DataFrame(columns=['cmdb_id','log_name','points_in_window','mean_value','p95_value'])

# 2b) Specifically report log.error_count entries in the window with earliest/latest and max value
if not df_log_window.empty:
    df_error_count_window = df_log_window[df_log_window['log_name'] == 'log.error_count'].copy()
    if not df_error_count_window.empty:
        error_count_details = (
            df_error_count_window
            .groupby('cmdb_id', as_index=False)
            .agg(
                points_in_window=('value','count'),
                earliest_ts=('timestamp','min'),
                latest_ts=('timestamp','max'),
                max_value=('value','max')
            )
            .sort_values('points_in_window', ascending=False)
            .reset_index(drop=True)
        )
        # format timestamps
        error_count_details['earliest_ts'] = pd.to_datetime(error_count_details['earliest_ts']).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
        error_count_details['latest_ts'] = pd.to_datetime(error_count_details['latest_ts']).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    else:
        error_count_details = pd.DataFrame(columns=['cmdb_id','points_in_window','earliest_ts','latest_ts','max_value'])
else:
    error_count_details = pd.DataFrame(columns=['cmdb_id','points_in_window','earliest_ts','latest_ts','max_value'])

# 3) Error logs within incident window
df_error_window = df_error[(df_error['timestamp'] >= start) & (df_error['timestamp'] <= end)].copy()
if not df_error_window.empty:
    err_grp = df_error_window.groupby('cmdb_id', sort=False)
    error_summary_window = err_grp.agg(
        count=('message','count'),
        earliest_ts=('timestamp','min'),
        latest_ts=('timestamp','max')
    ).reset_index()
    samples = err_grp['message'].apply(lambda s: " | ".join(s.astype(str).head(5).tolist())).reset_index(name='samples')
    error_summary_window = error_summary_window.merge(samples, on='cmdb_id', how='left')
    error_summary_window['earliest_ts'] = pd.to_datetime(error_summary_window['earliest_ts']).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    error_summary_window['latest_ts'] = pd.to_datetime(error_summary_window['latest_ts']).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    error_summary_window = error_summary_window.sort_values('count', ascending=False).head(10).reset_index(drop=True)
else:
    error_summary_window = pd.DataFrame(columns=['cmdb_id','count','earliest_ts','latest_ts','samples'])

# Display compact results (variables)
metric_global, metric_window_summary, log_window_summary, error_count_details, error_summary_window

```
Out[2]:
```
summary = (
    "Summary of findings (plain English):\n\n"
    "1) Global metrics (full series): carts, catalogue, orders, payment and user each have multiple KPIs (cpu, mem, socket, latency-50/90, workload, etc.).\n\n"
    "2) Incident-window comparisons (2024-01-20 19:35:40 to 20:05:40 UTC):\n"
    "- user.diskio: 25 points in window, 2 points above the global P95. Earliest above-P95 at 2024-01-20T19:38:00Z; max in window ≈ 59565.49 which is ~1.45× the global P95. This is the largest relative spike observed.\n"
    "- carts.socket: 25 points, 2 points above global P95. Earliest above-P95 at 2024-01-20T19:39:00Z; max ≈ 14.6667 (~1.07× P95).\n"
    "- orders.cpu and orders latency (latency-50 and latency-90): each had 25 points and 2 points above global P95 with earliest above-P95 timestamps around 19:40–19:54Z, and ratios ≈1.05–1.06. These indicate modest spikes in CPU and latency for orders.\n"
    "- carts.mem shows 2 points above P95 but the max is only marginally above P95 (ratio ≈1.00005).\n"
    "- Several other KPIs (catalogue.socket, payment.socket, user.socket, etc.) had data in-window but no values exceeding their global P95 in this window.\n\n"
    "3) Logs in the window: many services have log.row_count activity (front-end is highest). log.error_count is present for multiple cmdb_id values in the window but all aggregated values are 0 (max=0). Detailed per-service log.error_count entries show points in window and earliest/latest timestamps, but no nonzero error counts.\n\n"
    "4) Error logs: no error_log messages were recorded in the incident window (empty result).\n\n"
    "Interpretation / recommended next steps:\n"
    "- The most prominent anomaly is the user.diskio spike (max ≈1.45× P95) — investigate user service disk I/O and underlying storage. Also investigate carts.socket and orders (CPU and latency) spikes which may contribute to user-visible issues.\n"
    "- Since log.error_count is zero and there are no textual error logs in-window, use detailed traces, raw logs, and host-level I/O metrics for the user and carts services to pinpoint causes."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id    kpi_name  total_count   global_mean    global_p95
0    carts         cpu           25  1.438002e+00  1.519959e+00
1    carts  latency-50           25  2.556400e-02  2.787600e-02
2    carts  latency-90           25  5.499800e-02  6.838500e-02
3    carts         mem           25  2.128176e+08  2.129022e+08
4    carts      socket           25  1.273894e+01  1.368667e+01
..     ...         ...          ...           ...           ...
27    user  latency-50           25  6.064000e-03  7.688000e-03
28    user  latency-90           25  3.466700e-02  6.987600e-02
29    user         mem           25  1.167036e+07  1.459649e+07
30    user      socket           25  1.969267e+01  2.100000e+01
31    user    workload           25  1.835073e+01  1.909682e+01

[32 rows x 5 columns],       cmdb_id    kpi_name  points_in_window  count_above_p95 earliest_ts_above_p95  max_value_in_window  ratio_max_over_p95  total_count   global_mean    global_p95
0        user      diskio              25.0              2.0  2024-01-20T19:38:00Z         5.956549e+04            1.454797           25  4.429828e+03  4.094418e+04
1       carts      socket              25.0              2.0  2024-01-20T19:39:00Z         1.466667e+01            1.071603           25  1.273894e+01  1.368667e+01
2      orders         cpu              25.0              2.0  2024-01-20T19:40:00Z         1.243366e+00            1.064194           25  1.107163e+00  1.168364e+00
3      orders  latency-90              25.0              2.0  2024-01-20T19:54:00Z         9.426200e-02            1.051631           25  6.548500e-02  8.963400e-02
4      orders  latency-50              25.0              2.0  2024-01-20T19:54:00Z         4.743200e-02            1.046047           25  4.019500e-02  4.534400e-02
..        ...         ...               ...              ...                   ...                  ...                 ...          ...           ...           ...
27      carts         mem              25.0              2.0  2024-01-20T19:46:00Z         2.129118e+08            1.000045           25  2.128176e+08  2.129022e+08
28  catalogue      socket              25.0              1.0  2024-01-20T20:00:00Z         6.283333e+00            1.047222           25  6.011333e+00  6.000000e+00
29    payment      socket              25.0              0.0                   NaN         5.000000e+00            1.000000           25  5.000000e+00  5.000000e+00
30       user      socket              25.0              0.0                   NaN         2.100000e+01            1.000000           25  1.969267e+01  2.100000e+01
31     orders       error              25.0              0.0                   NaN         0.000000e+00                 NaN           25  0.000000e+00  0.000000e+00

[32 rows x 10 columns],          cmdb_id         log_name  points_in_window   mean_value  p95_value
0      front-end    log.row_count                25  1878.960000     2046.0
1      front-end  log.error_count                25     0.000000        0.0
2      catalogue    log.row_count                25   155.960000      167.8
3      catalogue  log.error_count                25     0.000000        0.0
4   queue-master  log.error_count                25     0.000000        0.0
5   queue-master    log.row_count                25   528.480000      583.2
6       shipping    log.row_count                25    58.720000       64.8
7       shipping  log.error_count                25     0.000000        0.0
8         orders  log.error_count                25     0.000000        0.0
9         orders    log.row_count                25   117.440000      129.6
10       payment  log.error_count                25     0.000000        0.0
11       payment    log.row_count                25    97.120000      104.8
12          user  log.error_count                25     0.000000        0.0
13          user    log.row_count                25   566.880000      613.6
14      carts-db  log.error_count                24     0.000000        0.0
15      carts-db    log.row_count                24     1.000000        1.0
16         carts  log.error_count                23     0.000000        0.0
17         carts    log.row_count                23    48.913043       75.0
18    session-db    log.row_count                 5     5.000000        5.0
19    session-db  log.error_count                 5     0.000000        0.0,         cmdb_id  points_in_window           earliest_ts             latest_ts  max_value
0     front-end                25  2024-01-20T19:38:00Z  2024-01-20T20:02:00Z          0
1     catalogue                25  2024-01-20T19:38:00Z  2024-01-20T20:02:00Z          0
2  queue-master                25  2024-01-20T19:38:00Z  2024-01-20T20:02:00Z          0
3      shipping                25  2024-01-20T19:38:00Z  2024-01-20T20:02:00Z          0
4        orders                25  2024-01-20T19:38:00Z  2024-01-20T20:02:00Z          0
5       payment                25  2024-01-20T19:38:00Z  2024-01-20T20:02:00Z          0
6          user                25  2024-01-20T19:38:00Z  2024-01-20T20:02:00Z          0
7      carts-db                24  2024-01-20T19:38:00Z  2024-01-20T20:01:00Z          0
8         carts                23  2024-01-20T19:38:00Z  2024-01-20T20:02:00Z          0
9    session-db                 5  2024-01-20T19:40:00Z  2024-01-20T20:00:00Z          0, Empty DataFrame
Columns: [cmdb_id, count, earliest_ts, latest_ts, samples]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Ensure metric df is loaded and timestamps are parsed (reuse existing variable if present)
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure timestamp dtype and numeric value
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# helper: global p95
def series_p95(s):
    arr = s.dropna().to_numpy(dtype=float)
    return float(np.nanpercentile(arr, 95)) if arr.size > 0 else np.nan

# Candidate KPIs as tuples (cmdb_id, kpi_name)
candidates = [
    ('user', 'diskio'),
    ('carts', 'socket'),
    ('orders', 'cpu'),
    ('orders', 'latency-50'),
    ('orders', 'latency-90'),
    ('carts', 'mem')
]

# 1) Compute global P95 for each candidate using full series
global_stats = []
for cid, kpi in candidates:
    sel = df_metric[(df_metric['cmdb_id'] == cid) & (df_metric['kpi_name'] == kpi)]
    gp95 = series_p95(sel['value'])
    gmean = float(sel['value'].mean()) if sel['value'].count() > 0 else np.nan
    total_count = int(sel['value'].count())
    global_stats.append({
        'cmdb_id': cid,
        'kpi_name': kpi,
        'total_count': total_count,
        'global_mean': gmean,
        'global_p95': gp95
    })
global_stats = pd.DataFrame(global_stats)

# 2) Filter metric series to the incident window (inclusive)
start = pd.to_datetime("2024-01-20 19:35:40", utc=True)
end   = pd.to_datetime("2024-01-20 20:05:40", utc=True)
mask_window = (df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)
df_window = df_metric[mask_window].copy()

# 3) Identify consecutive sub-series where value > global_P95 for each candidate
fault_rows = []
for _, row in global_stats.iterrows():
    cid = row['cmdb_id']
    kpi = row['kpi_name']
    gp95 = row['global_p95']
    if np.isnan(gp95):
        continue  # no threshold to compare
    # subset windowed data for this series
    sub = df_window[(df_window['cmdb_id'] == cid) & (df_window['kpi_name'] == kpi)].copy()
    if sub.empty:
        continue
    sub = sub.sort_values('timestamp').reset_index(drop=True)
    # mark points above global p95
    sub['above'] = sub['value'] > gp95
    # keep only points that are above
    above_df = sub[sub['above']].copy()
    if above_df.empty:
        continue
    # identify consecutive runs: difference in timestamp == 60 seconds -> same run
    above_df['prev_ts'] = above_df['timestamp'].shift(1)
    above_df['diff_s'] = (above_df['timestamp'] - above_df['prev_ts']).dt.total_seconds()
    # start of a new run where diff_s != 60 or prev is NaT
    above_df['new_run'] = (above_df['diff_s'] != 60) | (above_df['prev_ts'].isna())
    above_df['run_id'] = above_df['new_run'].cumsum()
    # aggregate runs
    runs = above_df.groupby('run_id', sort=False).agg(
        cmdb_id=('cmdb_id', 'first'),
        kpi_name=('kpi_name', 'first'),
        fault_start_ts=('timestamp', 'min'),
        fault_end_ts=('timestamp', 'max'),
        points_above_p95=('value', 'count'),
        max_value_in_fault=('value', 'max')
    ).reset_index(drop=True)
    # compute minutes_length (number of consecutive points)
    runs['minutes_length'] = runs['points_above_p95']  # per-minute sampling assumption
    runs['global_p95'] = gp95
    # ratio
    runs['max_over_p95_ratio'] = runs['max_value_in_fault'] / gp95
    # format timestamps to UTC string
    runs['fault_start_ts'] = runs['fault_start_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    runs['fault_end_ts'] = runs['fault_end_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    # apply rule 4: exclude single-point runs unless ratio >= 1.5
    def keep_run(r):
        if r['points_above_p95'] > 1:
            return True
        # single point
        return (r['max_over_p95_ratio'] >= 1.5)
    runs = runs[runs.apply(keep_run, axis=1)].copy()
    # reorder columns as requested
    runs = runs[['cmdb_id','kpi_name','fault_start_ts','fault_end_ts','minutes_length',
                 'points_above_p95','max_value_in_fault','global_p95','max_over_p95_ratio']]
    fault_rows.append(runs)

# concat all fault rows and sort/limit
if fault_rows:
    faults_df = pd.concat(fault_rows, ignore_index=True)
    # numeric rounding for compactness
    faults_df['max_value_in_fault'] = faults_df['max_value_in_fault'].round(6)
    faults_df['global_p95'] = faults_df['global_p95'].round(6)
    faults_df['max_over_p95_ratio'] = faults_df['max_over_p95_ratio'].round(6)
    # sort by points_above_p95 desc then ratio desc
    faults_df = faults_df.sort_values(['points_above_p95','max_over_p95_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)
else:
    faults_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_ts','fault_end_ts','minutes_length',
        'points_above_p95','max_value_in_fault','global_p95','max_over_p95_ratio'
    ])

# Display the compact faults table
faults_df

```
Out[3]:
```
```python
summary = (
    "Summary of fault candidate sub-series (incident window 2024-01-20 19:35:40 to 20:05:40 UTC):\n\n"
    "1) user - diskio: a 2-minute consecutive run from 2024-01-20T19:38:00Z to 2024-01-20T19:39:00Z (2 points). "
    "Max value = 59565.485792, global P95 = 40944.180657, max_over_p95_ratio ≈ 1.4548. This is the largest relative spike but <1.5 for single-point threshold — included because it is 2 consecutive points.\n\n"
    "2) orders - latency-90: a 2-minute run from 2024-01-20T19:54:00Z to 2024-01-20T19:55:00Z (2 points). "
    "Max value = 0.094262, global P95 = 0.089634, ratio ≈ 1.0516 (modest exceedance).\n\n"
    "3) orders - latency-50: a 2-minute run from 2024-01-20T19:54:00Z to 2024-01-20T19:55:00Z (2 points). "
    "Max value = 0.047432, global P95 = 0.045344, ratio ≈ 1.0461 (modest exceedance).\n\n"
    "No qualifying fault runs were found for the other candidates (carts-socket, orders-cpu, carts-mem) within the window under the applied rules. "
    "Note: single isolated points below the 1.5× ratio threshold were excluded by the rules."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name        fault_start_ts          fault_end_ts  minutes_length  points_above_p95  max_value_in_fault    global_p95  max_over_p95_ratio
0    user      diskio  2024-01-20T19:38:00Z  2024-01-20T19:39:00Z               2                 2        59565.485792  40944.180657            1.454797
1  orders  latency-90  2024-01-20T19:54:00Z  2024-01-20T19:55:00Z               2                 2            0.094262      0.089634            1.051629
2  orders  latency-50  2024-01-20T19:54:00Z  2024-01-20T19:55:00Z               2                 2            0.047432      0.045344            1.046058```
```